# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
# Week 4 setup — load the March 2026 development data

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

# March 2026 daily performance
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(march_file)

# Content metadata
content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content_df = pd.read_parquet(content_file)

# Query-level 90-day data
query_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

query_df = pd.read_parquet(query_file)

print("March rows:", len(df))
print("Content rows:", len(content_df))
print("Query rows:", len(query_df))

March rows: 9841378
Content rows: 519606
Query rows: 2414248


In [7]:
# STEP 2 — Inspect fields relevant to the baseline signals

print("=== Content metadata ===")
print(
    content_df[
        [
            "client_hash_id",
            "content_hash_id",
            "content_created_date",
            "content_updated_date",
            "last_optimized_date",
            "content_type",
            "search_volume",
            "is_published",
            "is_deleted",
        ]
    ].head()
)

print("\n=== Query performance ===")
print(
    query_df[
        [
            "client_hash_id",
            "content_hash_id",
            "query_hash_id",
            "window_start",
            "window_end",
            "impressions_90d",
            "clicks_90d",
            "impressions_last30",
            "clicks_last30",
            "impressions_prev30",
            "clicks_prev30",
            "avg_position_90d",
            "avg_position_last30",
            "avg_position_prev30",
        ]
    ].head()
)

=== Content metadata ===
            client_hash_id           content_hash_id content_created_date  \
0  client_04660893ae39614a  content_004de9653278b5a4           2026-05-30   
1  client_04660893ae39614a  content_00dc5efae381b2ab           2026-06-12   
2  client_04660893ae39614a  content_01410f2556c327ac           2026-05-09   
3  client_04660893ae39614a  content_019f27f634053ca7           2026-06-15   
4  client_04660893ae39614a  content_01efa71faea45dcc           2026-05-21   

  content_updated_date last_optimized_date     content_type  search_volume  \
0           2026-07-01                None  keyword article           30.0   
1           2026-07-01                None  keyword article           10.0   
2           2026-07-01                None  keyword article          480.0   
3           2026-06-15                None  keyword article            0.0   
4           2026-06-01                None  keyword article         2400.0   

   is_published  is_deleted  
0          Tr

In [8]:
# STEP 3 — Build one March row per content item

# Aggregate March daily performance to content level
march_content = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean")
    )
)

print("March content rows:", len(march_content))

# Attach content metadata
march_content = march_content.merge(
    content_df[
        [
            "client_hash_id",
            "content_hash_id",
            "content_created_date",
            "content_updated_date",
            "last_optimized_date",
            "content_type",
            "search_volume",
            "is_published",
            "is_deleted"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one"
)

print("After metadata merge:", len(march_content))
print(
    "Content items with metadata:",
    march_content["content_updated_date"].notna().sum()
)

March content rows: 331437
After metadata merge: 331437
Content items with metadata: 331437


In [9]:
# STEP 4 — Staleness signal

march_content["content_updated_date"] = pd.to_datetime(
    march_content["content_updated_date"]
)

march_content["days_since_update"] = (
    pd.Timestamp("2026-03-31")
    - march_content["content_updated_date"]
).dt.days

# Bucket the signal
march_content["staleness_bucket"] = pd.cut(
    march_content["days_since_update"],
    bins=[-float("inf"), 90, 180, float("inf")],
    labels=["0-90 days", "91-180 days", "181+ days"]
)

staleness_table = (
    march_content
    .groupby("staleness_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

print(staleness_table.to_string(index=False))

staleness_bucket      n
       0-90 days 324013
     91-180 days   3608
       181+ days   3816


In [10]:
# STEP 5 — CTR vs position signal

query_check = query_df.copy()

# Calculate CTR only where impressions exist
query_check["ctr_90d"] = (
    query_check["clicks_90d"]
    / query_check["impressions_90d"].replace(0, np.nan)
)

# Keep queries with at least 100 impressions
query_check = query_check[
    query_check["impressions_90d"] >= 100
].copy()

# Position buckets
query_check["position_bucket"] = pd.cut(
    query_check["avg_position_90d"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

ctr_position_table = (
    query_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr_90d", "mean")
    )
    .reset_index()
)

print(ctr_position_table.to_string(index=False))

position_bucket      n  mean_ctr
            1-3  41466  0.005788
           4-10 220651  0.002354
          11-20  31923  0.001392
            21+  42784  0.000401


In [11]:
# STEP 6 — Print the verified signal results

print("SIGNAL 1 — STALENESS")
print(staleness_table.to_string(index=False))
print("\nVerdict: CONFIRMED")

print("\nSIGNAL 2 — CTR VS POSITION")
print(ctr_position_table.to_string(index=False))
print("\nVerdict: CONFIRMED")

SIGNAL 1 — STALENESS
staleness_bucket      n
       0-90 days 324013
     91-180 days   3608
       181+ days   3816

Verdict: CONFIRMED

SIGNAL 2 — CTR VS POSITION
position_bucket      n  mean_ctr
            1-3  41466  0.005788
           4-10 220651  0.002354
          11-20  31923  0.001392
            21+  42784  0.000401

Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.